# Learning Objectives

* Build a per-sample transposon-sequencing (Tn-seq) workflow from Python wrapper scripts and an Apptainer container.
* Generate and register FASTQ inputs, references, support files (Je, R scripts), a container, and a workflow DAG through the Pegasus Python API.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the results: deduplicated alignments, BigWig tracks, and per-gene insertion count tables.

# TNseq Alignment and Annotation Workflow

This workflow maps bacterial Tn-seq reads to the *Caulobacter crescentus* NA1000 genome and counts
transposon insertions per gene, converted from the chienlab-tnseq Snakemake workflow. Each sample is
UMI-clipped, filtered for the transposon sequence, aligned, deduplicated and assigned to genome positions in
parallel; the per-sample count tables are then concatenated into the final TSVs. See `README.md` in this
repository for the full pipeline description and outputs.

**Workflow Jobs:**
1. `clip` - clip UMIs with Je
2. `seqkit_grep` - keep reads carrying the transposon sequence
3. `bwa_mem` - align reads with BWA-MEM
4. `rm_dupe` - remove PCR duplicates with Je
5. `genomecov` - assign genome positions with bedtools
6. `bedtools_map` - map counts to gene features (total/unique × mid/full)
7. `bam2bw` - BigWig coverage tracks
8. `tab_generate` - per-sample count tables
9. `concat` - concatenate all samples into the final TSVs
10. `seqkit_qc` - FASTQ and BAM QC statistics

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment.

## 1. Prepare Sample Input Data

Two small test samples (100K reads each, from NCBI SRA) ship in `data/test/`, and the reference genome
(with its BWA index) and gene BED files ship in `references/`. Samples are auto-discovered from the
`*.fq.gz` files in the FASTQ directory:

In [ ]:
!ls -l data/test/ references/NC_011916.fasta* references/CCNA_mid_trim10_10.bed references/CCNA_genes.bed

## 2. Build the Apptainer Container

The workflow's jobs run inside an Apptainer container (`Apptainer/Tnseq_Container.def`) that bundles BWA,
samtools, bedtools, seqkit, deepTools, Java (for Je) and R. Building it can take a few minutes; skip the
rebuild if the `.sif` already exists.

In [ ]:
%%bash
if [ ! -f Apptainer/Tnseq_Container.sif ]; then
    apptainer build Apptainer/Tnseq_Container.sif Apptainer/Tnseq_Container.def
else
    echo "Container already built: Apptainer/Tnseq_Container.sif"
fi

## 3. Generate the Workflow

The following cell uses `workflow_generator.py` — the same generator used by the command-line tool — to
build the Pegasus workflow DAG (site, transformation, and replica catalogs) and write it to `workflow.yml`.
The inputs are parsed by the CLI's own argument parser (`build_parser()`), so nothing about the workflow is
defined here: change the pipeline in `workflow_generator.py`, and both the notebook and the CLI pick it up.

`create_sites_catalog()` writes a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import build_parser, discover_samples, workflow_from_args

EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

# The same arguments the CLI takes (see README.md)
args = build_parser().parse_args([
    "--fastq-dir", "data/test/",
    "--ref-fasta", "references/NC_011916.fasta",
    "--ref-mid", "references/CCNA_mid_trim10_10.bed",
    "--ref-full", "references/CCNA_genes.bed",
    "--output", "workflow.yml",
])
args.samples = args.samples or discover_samples(args.fastq_dir)
print(f"Samples: {args.samples}")

workflow = workflow_from_args(args)

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE,
                                       container_sif=args.container_sif)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating TNseq workflow DAG...")
workflow.create_workflow()

workflow.write()
print("\nTNseq Workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. The graph shows one
clip → seqkit_grep → bwa_mem → rm_dupe chain per sample, the four `bedtools_map` variants and `tab_generate`
jobs per sample, and the `concat` and QC jobs that gather every sample.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the site chosen above (`EXEC_SITE`).

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and
prints the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/`:

| Output | Description |
|--------|-------------|
| `bwa_aln/{sample}_marked.bam`, `.bam.bai` | Deduplicated alignments |
| `bwa_aln/dedup_metrics/{sample}_metrics.txt` | Je deduplication metrics |
| `bigwig/{sample}.bw` | BigWig coverage tracks |
| `read_counts/{total,unique}counts_{mid,full}/{sample}.*.tab` | Per-sample insertion counts per gene |
| `read_counts/{total,unique}counts_{mid,full}.tsv` | All samples concatenated |
| `QC/QC_fq.txt`, `QC/QC_bam.txt` | FASTQ and BAM statistics |

In [ ]:
!ls -ltR output/

Let's look at the unique insertion counts per gene (mid-gene features) across the samples:

In [ ]:
import pandas as pd

pd.read_csv("output/read_counts/uniquecounts_mid.tsv", sep="\t").head(10)